In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv


# データ読み込み

In [2]:
#"Id"列をインデックスに指定
import pandas as pd
test = pd.read_csv("/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv").set_index("Id")
train = pd.read_csv("/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv").set_index("Id")

#目的変数”SalePrice"を取り出す
y = train.SalePrice
X = train.drop(columns=["SalePrice"])

# 数値列とカテゴリ列に分類

In [3]:
#数値列
num_cols = [col for col in X.columns
            if X[col].dtype in ["int64","float64"]]

#カテゴリ列
cat_cols = [col for col in X.columns
           if X[col].dtype in ["category","object"]]

#2つ合わせた特徴量の数は、SalePriceとId列を除いた79列
len(num_cols + cat_cols)

79

# 前処理
簡単な前処理のパイプラインを作る。

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

#数値列の前処理。基本的な平均値埋め
numerical_transformer = Pipeline(
    steps = [
        ("imputer", SimpleImputer(strategy = "mean"))
    ]
)

#カテゴリ列の前処理。最頻値埋め
categorical_transformer = Pipeline(
    steps = [
        ("imputer", SimpleImputer(strategy = "most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]
)

preprocessor = ColumnTransformer(
    transformers = [
        ("num", numerical_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ]
)


# model作成
今回は、目的変数SalePriceを予測する回帰モデルなので、まずはRandomForestRegressorで予測してみる。

In [5]:
from sklearn.ensemble import RandomForestRegressor

#前処理+モデルの入った基本のベースライン
RFR_pipeline = Pipeline(
    steps = [
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(
            n_jobs = -1,
            random_state = 10,
            n_estimators = 300
        ))
    ]
)

# CV検証

In [6]:
from sklearn.model_selection import KFold, cross_val_score

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=10
)

cv_scores = -cross_val_score(
    RFR_pipeline,
    X,y,cv=cv,
    scoring="neg_root_mean_squared_log_error",
    n_jobs=-1
)

print("各FoldのRMSLE:", cv_scores)
print("平均RMSLE:", cv_scores.mean())
print("標準偏差:", cv_scores.std())

各FoldのRMSLE: [0.14700779 0.14775879 0.13806277 0.16630975 0.13365781]
平均RMSLE: 0.1465593812574692
標準偏差: 0.01122817847932152


# 提出物の作成

In [7]:
RFR_pipeline.fit(X,y)
test_preds = RFR_pipeline.predict(test)
submission = pd.DataFrame({
    "Id":test.index,
    "SalePrice":test_preds
})

In [8]:
submission.to_csv("submission.csv",index=False)

# Baselineの結果・考察

## 結果

RandomForestRegressorを使用して、基本的な前処理のみを行ったBaselineモデルを作成した。

- 5-fold CV : 0.1466
- Kaggle Public Score : 0.14629
- 順位 : 2639 / 3808
- 上位約69%

## 考察

CVスコアとPublic Scoreが非常に近い結果となった。

このことから、今回使用したCVによる検証方法は、
未知データに対する性能をある程度適切に推定できていると考えられる。

また、今回は基本的な前処理とRandomForestRegressorのみを使用しており、
複雑な特徴量作成やハイパーパラメータチューニングは行っていない。

そのため、今回のCVスコア 0.1466 を今後のモデル改善における
基準スコア（Baseline）として使用する。

## 今後の方針

今後は以下の順番で、CVスコアが改善するか検証する。

1. Preprocesseing 前処理改善
2. Feature Engineering EDAをもとにした特徴量エンジニアリング
3. Model Comparoson 複数モデルの比較
4. Hyperparameter Tuning 有望なモデルのハイパーパラメータ調整
5. Ensemble 必要に応じて複数モデルのアンサンブル

各変更の前後でCVスコアを比較し、
どの処理が予測性能の改善に貢献したのか記録していく。